## boosting

In [7]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

data_train = pd.read_csv('../data/train/train.csv')
data_test = pd.read_csv('../data/test/test.csv')

In [18]:
copy_data_train = data_train.copy()        



copy_data_train['CabinKnown'] = copy_data_train['Cabin'].notna()
copy_data_train['Age'] = copy_data_train['Age'].fillna(copy_data_train['Age'].median())
copy_data_train['Embarked'] = copy_data_train['Embarked'].fillna('S')

copy_data_train['CabinKnown'] = copy_data_train['CabinKnown'].astype('int32')
replace_dict = {'male': 0, 'female': 1}
copy_data_train['Sex'] = copy_data_train['Sex'].replace(replace_dict)
copy_data_train['Sex'] = copy_data_train['Sex'].astype('int32')

copy_data_train.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,CabinKnown
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,S,1
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,S,0


In [19]:
from sklearn.preprocessing import OneHotEncoder


encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

# data_procced_categories['Embarked'] = encoder.fit_transform(data_procced_categories['Embarked'])
encoded = encoder.fit_transform(copy_data_train[['Embarked']],)
encoded_df= pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(["Embarked"]),
    index=copy_data_train.index,
)

copy_data_train = pd.concat(
    [
        copy_data_train.drop(columns=['Embarked']),
        encoded_df
    ],
    axis=1
)
copy_data_train


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,CabinKnown,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,A/5 21171,7.2500,NaN,0,0.0,1.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,PC 17599,71.2833,C85,1,0.0,0.0
2,3,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,STON/O2. 3101282,7.9250,NaN,0,0.0,1.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,113803,53.1000,C123,1,0.0,1.0
4,5,0,3,"Allen, Mr. William Henry",0,35.0,0,0,373450,8.0500,NaN,0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",0,27.0,0,0,211536,13.0000,NaN,0,0.0,1.0
887,888,1,1,"Graham, Miss. Margaret Edith",1,19.0,0,0,112053,30.0000,B42,1,0.0,1.0
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",1,28.0,1,2,W./C. 6607,23.4500,NaN,0,0.0,1.0
889,890,1,1,"Behr, Mr. Karl Howell",0,26.0,0,0,111369,30.0000,C148,1,0.0,0.0


### xgboost

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
import xgboost as xgb

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# model = xgb.XGBClassifier(n_estimators=200, learning_rate=0.4, max_depth=5)
fold_val_metric = []
fold_train_metric = []
model = xgb.XGBClassifier(eval_metric="error")

param_distributions = {
    "n_estimators": [50, 100, 150, 200, 300],
    "learning_rate": [0.09, 0.08, 0.1, 0.05],
    "max_depth": [2, 3, 4, 5],
    "min_child_weight": [1, 2, 3, 5],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
}

random_scv = RandomizedSearchCV(estimator=model, random_state=42, param_distributions=param_distributions, n_iter=50, scoring='accuracy', cv=5, n_jobs=-1, return_train_score=True, verbose=2)

train_X = copy_data_train.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin', 'PassengerId'])
train_y = copy_data_train['Survived']
# print(train_X)
for train_indices, val_indices in skf.split(train_X, train_y):

    x_train_fold = train_X.iloc[train_indices]
    x_val_fold = train_X.iloc[val_indices]

    y_train_fold = train_y.iloc[train_indices]
    y_val_fold = train_y.iloc[val_indices]
    random_scv.fit(x_train_fold, y_train_fold)

    y_train_pred = random_scv.predict(x_train_fold)
    y_pred = random_scv.predict(x_val_fold)

    accuracy_train = accuracy_score(y_train_fold, y_train_pred)
    accuracy = accuracy_score(y_val_fold, y_pred)
    fold_val_metric.append(accuracy)
    fold_train_metric.append(accuracy_train)


print(f'best estimator: {random_scv.best_estimator_}')
print(f'best_params: {random_scv.best_params_}')

print(f"Mean val accuracy: {sum(fold_val_metric) / len(fold_val_metric):.4f}")
print(f"Mean train accuracy: {sum(fold_train_metric) / len(fold_train_metric):.4f}")
print(fold_val_metric)


# Mean val accuracy: 0.8361 n_estimators=100, learning_rate=0.1, max_depth=3
# Mean train accuracy: 0.8808

# best_params: {'subsample': 1.0, 'n_estimators': 150, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.08, 'colsample_bytree': 0.9}
# Mean val accuracy: 0.8361
# Mean train accuracy: 0.9032


Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=4, min_child_weight=5, n_estimators=50, subsample=0.7; total time=   0.1s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=4, min_child_weight=5, n_estimators=50, subsample=0.7; total time=   0.1s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=4, min_child_weight=5, n_estimators=50, subsample=0.7; total time=   0.1s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=4, min_child_weight=5, n_estimators=50, subsample=0.7; total time=   0.1s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=4, min_child_weight=5, n_estimators=50, subsample=0.7; total time=   0.1s
[CV] END colsample_bytree=0.9, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=200, subsample=0.7; total time=   0.0s
[CV] END colsample_bytree=0.9, learning_rate=0.05, max_depth=5, min_child_weight=1, n_estimators=200, subsample=0.7; total time=   0

## lightGBM

In [ ]:
import lightgbm as lgb

fold_val_metric = []
fold_train_metric = []


model = lgb.LGBMClassifier(
    learning_rate=0.1,
    verbose=-1
)

# param_distributions = {
#     "n_estimators": [50, 100, 150, 200, 300],
#     "learning_rate": [0.09, 0.08, 0.1, 0.05],
#     "max_depth": [2, 3, 4, 5],
#     "min_child_weight": [1, 2, 3, 5],
#     "subsample": [0.7, 0.8, 0.9, 1.0],
#     "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
# }

# random_scv = RandomizedSearchCV(estimator=model, random_state=42,
#                                  param_distributions=param_distributions, n_iter=50,
#                                    scoring='accuracy', cv=5, n_jobs=-1, 
#                                    return_train_score=True, )
                                   
# best estimator: LGBMClassifier(colsample_bytree=0.9, learning_rate=0.09, max_depth=5,
#                min_child_weight=2, subsample=0.7, verbose=-1)
# best_params: {'subsample': 0.7, 'n_estimators': 100, 'min_child_weight': 2, 'max_depth': 5, 'learning_rate': 0.09, 'colsample_bytree': 0.9}

for train_indices, val_indices in skf.split(train_X, train_y):

    model = lgb.LGBMClassifier(
        colsample_bytree=0.9,
        learning_rate=0.09,
        max_depth=5,
        min_child_weight=2, 
        subsample=0.7
    )
    x_train_fold = train_X.iloc[train_indices]
    x_val_fold = train_X.iloc[val_indices]

    y_train_fold = train_y.iloc[train_indices]
    y_val_fold = train_y.iloc[val_indices]
    model.fit(x_train_fold, y_train_fold)


    y_train_pred = model.predict(x_train_fold)
    y_pred = model.predict(x_val_fold)


    accuracy_train = accuracy_score(y_train_fold, y_train_pred)
    accuracy = accuracy_score(y_val_fold, y_pred)
    fold_val_metric.append(accuracy)
    fold_train_metric.append(accuracy_train)

# print(f'best estimator: {random_scv.best_estimator_}')
# print(f'best_params: {random_scv.best_params_}')


print(f"Mean val accuracy: {sum(fold_val_metric) / len(fold_val_metric):.4f}")
print(f"Mean train accuracy: {sum(fold_train_metric) / len(fold_train_metric):.4f}")
print(fold_val_metric)

# best estimator: LGBMClassifier(colsample_bytree=0.9, learning_rate=0.09, max_depth=5,
#                min_child_weight=2, subsample=0.7, verbose=-1)
# best_params: {'subsample': 0.7, 'n_estimators': 100, 'min_child_weight': 2, 'max_depth': 5, 'learning_rate': 0.09, 'colsample_bytree': 0.9}
# TODO: WHEN USING STRATIFIED K FOLD
# Mean val accuracy: 0.8372
# Mean train accuracy: 0.8995

Mean val accuracy: 0.8372
Mean train accuracy: 0.8995
[0.8603351955307262, 0.8595505617977528, 0.7921348314606742, 0.8426966292134831, 0.8314606741573034]


### catboost

In [ ]:
from catboost import CatBoostClassifier

fold_val_metric = []
fold_train_metric = []


model = CatBoostClassifier(verbose=0)

param_distributions = {
    "iterations": [50, 100, 150, 200],
    "learning_rate": [0.09, 0.08, 0.1, 0.05],
    "depth": [3, 4],
    "l2_leaf_reg": [1, 3, 5, 7, 10],
    "random_strength": [0, 0.5, 1, 2],
    "rsm": [0.7, 0.8, 0.9, 1.0],
}

catboost_csv = RandomizedSearchCV(estimator=model, random_state=42,
                                 param_distributions=param_distributions, n_iter=50,
                                   scoring='accuracy', cv=skf, n_jobs=-1, 
                                   return_train_score=True, error_score='raise')
                   




catboost_csv.fit(train_X, train_y)

# y_train_pred = catboost_csv.predict(train_X)
y_pred = catboost_csv.predict(train_X)

# for train_indices, val_indices in skf.split(train_X, train_y):

#     x_train_fold = train_X.iloc[train_indices]
#     x_val_fold = train_X.iloc[val_indices]

#     y_train_fold = train_y.iloc[train_indices]
#     y_val_fold = train_y.iloc[val_indices]
#     catboost_csv.fit(x_train_fold, y_train_fold)


#     y_train_pred = catboost_csv.predict(x_train_fold)
#     y_pred = catboost_csv.predict(x_val_fold)


#     accuracy_train = accuracy_score(y_train_fold, y_train_pred)
#     accuracy = accuracy_score(y_val_fold, y_pred)
#     fold_val_metric.append(accuracy)
#     fold_train_metric.append(accuracy_train)

print(f'best estimator: {catboost_csv.best_estimator_}')
print(f'best_params: {catboost_csv.best_params_}')
print(f'best score: {catboost_csv.best_score_:.4f}')


# print(f"Mean val accuracy: {sum(fold_val_metric) / len(fold_val_metric):.4f}")
# print(f"Mean train accuracy: {sum(fold_train_metric) / len(fold_train_metric):.4f}")
# print(fold_val_metric)

# из коробки
# Mean val accuracy: 0.8249
# Mean train accuracy: 0.8429


# best estimator: CatBoostClassifier(depth=3, iterations=200, l2_leaf_reg=7, learning_rate=0.08, random_strength=0.5, rsm=0.9, verbose=0)
# best_params: {'rsm': 0.9, 'random_strength': 0.5, 'learning_rate': 0.08, 'l2_leaf_reg': 7, 'iterations': 200, 'depth': 3}
# best params: 0.8417




best estimator: CatBoostClassifier(depth=3, iterations=200, l2_leaf_reg=7, learning_rate=0.1, random_strength=1, rsm=1.0, verbose=0)
best_params: {'rsm': 1.0, 'random_strength': 1, 'learning_rate': 0.1, 'l2_leaf_reg': 7, 'iterations': 200, 'depth': 3}
best score: 0.8384
